# Alignment and principal component analysis (PCA)

This notebook starts with the alignment of a protein trajectory to a reference geometry, by minimizing the root-mean-square deviation (RMSD).
This removes the (random) global rotation and translation of the protein, which is needed before analyzing the principal components that describe the collective motions of the protein.

This notebook shows how to perform PCA with bare NumPy operations, to let you better understand each step.
The MDTraj documentation also explains how to perform the [PCA analysis with scikit-learn](https://www.mdtraj.org/development/examples/pca.html).

The analysis uses the output files `init.pdb`, `scalars.csv` and `traj.dcd` of the villin headpiece simulation in [../04_protein/01_villin_headpiece.ipynb](../04_protein/01_villin_headpiece.ipynb).
These files must first be copied to this directory, as explained at the end of that notebook.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import mdtraj
import nglview
import numpy as np
import pandas
from openmm import app
from pdbfixer import PDBFixer

The next cell loads the trajectory and strips water molecules, because all analyses below only concern the protein.

In [ ]:
traj = mdtraj.load("traj.dcd", top="init.pdb")
traj.restrict_atoms(traj.topology.select("protein"))
df = pandas.read_csv("scalars.csv")

## 1. RMSD alignment

We will use the [superpose](https://www.mdtraj.org/development/api/generated/mdtraj.Trajectory.superpose.html) method from MDTraj to remove the global rotation and translation of the protein.
All frames will be aligned with the first frame of the trajectory, by setting the second argument to zero.
Note that `superpose` modifies the trajectory in place.

In [ ]:
traj.superpose(traj, 0)

This was relatively easy.
We can also (optionally) compute the RMSD with respect to the first frame as a function of time, with the [rmsd](https://www.mdtraj.org/development/api/generated/mdtraj.rmsd.html) function:

In [ ]:
rmsds = mdtraj.rmsd(traj, traj, 0)
plt.close("rmsd")  # This is needed to rerun the code cell correctly
fig, ax = plt.subplots(num="rmsd")
ax.plot(df["Time (ps)"], rmsds)
ax.set_xlabel("Time [ps]")
ax.set_ylabel("RMSD [nm]")

The first frame of the trajectory is very close to the minimized structure, in which the atoms are still at rest.
The rapid initial increase of the RMSD is caused by the thermal motion of the atoms, which builds up while the thermostat heats up the system.
After this, the protein geometry slowly drifts away from its initial structure.

In the following visualization, the effect of the alignment is clear: the protein no longer slowly rotates and translates, as it did in [01_internal_coordinates_averages.ipynb](01_internal_coordinates_averages.ipynb).
Instead, it just seems to wiggle in place.

In [ ]:
view = nglview.show_mdtraj(traj)
view.add_representation("ball+stick", selection="protein")
view

## 2. The covariance matrix

For the remaining part of the analysis, the initialization phase is removed from the trajectory.
The first 250 frames, i.e. the first $5\,\mathrm{ps}$, are discarded, in line with the warm-up phase observed in [01_internal_coordinates_averages.ipynb](01_internal_coordinates_averages.ipynb).

The first step in a principal component analysis is to compute the covariance matrix of the atomic positions, for which we will use NumPy's [cov](https://numpy.org/doc/stable/reference/generated/numpy.cov.html) function.
In this notebook, all atoms are included, but one may also work on a subset, e.g. the alpha carbons, instead.

Because this is a very short MD simulation of a rather rigid protein, a direct visualization of the covariance does not show much.
To keep the plot legible, only the part of the covariance matrix involving the alpha carbon atoms is shown.

In [ ]:
# We first need to change the shape of the array with atomic positions,
# such that each row contains all Cartesian coordinates of one frame.
print(traj.xyz.shape)
xyz = traj.xyz.reshape((-1, traj.n_atoms * 3))
print(xyz.shape)

# The first 250 frames are skipped because they cover the initialization phase.
nskip = 250
xyz_eq = xyz[nskip:]
time_eq = df["Time (ps)"][nskip:]

# rowvar=False indicates that every column is one variable (Cartesian coordinate)
# and every row is one observation (frame).
covar = np.cov(xyz_eq, rowvar=False)
print(covar.shape)

# Select the alpha carbon atoms
selection = traj.top.select("name CA")

# Use the selection to slice the covariance matrix:
# the X, Y and Z coordinates of each selected atom are retained.
selection_xyz = np.array([3 * selection, 3 * selection + 1, 3 * selection + 2]).T.ravel()
sub_covar = covar[selection_xyz][:, selection_xyz]

# Show the covariance matrix.
# The color scale is clipped at half of the largest value, to make smaller covariances visible.
plt.close("covar")  # This is needed to rerun the code cell correctly
fig, ax = plt.subplots(num="covar")
absvmax = abs(sub_covar).max() / 2
m = ax.matshow(sub_covar, vmin=-absvmax, vmax=absvmax, cmap="PiYG")
ax.set_xlabel("Cartesian coordinate index i (alpha carbons)")
ax.set_ylabel("Cartesian coordinate index j (alpha carbons)")
fig.colorbar(m, label="Covariance [nm$^2$]")

The diagonal of the covariance matrix contains the variances of the Cartesian coordinates.
The root-mean-square fluctuation (RMSF) of an atom is the square root of the sum of the variances of its X, Y and Z coordinates.
A high RMSF corresponds to a more flexible atom.
To make the RMSF plot legible, the following example only visualizes the RMSF for the alpha carbon atoms.

In [ ]:
# The diagonal elements corresponding to X, Y and Z coordinates
# of each atom are summed, such that one value per alpha
# carbon is retained.
rmsf = np.sqrt(np.diag(covar)[selection_xyz].reshape(-1, 3).sum(axis=1))
plt.close("rmsf")  # This is needed to rerun the code cell correctly
fig, ax = plt.subplots(num="rmsf")
ax.plot(rmsf)
ax.set_xlabel("Alpha carbon index")
ax.set_ylabel("RMSF")

A common result for this plot is that the termini are more flexible than the central part of the protein chain.
Also flexible loops can be identified with this analysis.
(Keep in mind that this example is based on a relatively short MD simulation, such that the results may not yet be converged.)

One can also compute the RMSF directly, without first computing the whole covariance matrix and then extracting only the diagonal elements, as shown in the following example.

In [ ]:
# By default, np.cov divides by N-1, while np.var divides by N.
# With ddof=1, np.var also divides by N-1, to obtain identical results.
rmsf_alt = np.sqrt(np.var(xyz_eq, axis=0, ddof=1)[selection_xyz].reshape(-1, 3).sum(axis=1))
print(rmsf[:5])
print(rmsf_alt[:5])

## 3. Diagonalization of the covariance matrix

The next step is to obtain the eigenvalues and eigenvectors, for which we use NumPy's [eigh](https://numpy.org/doc/stable/reference/generated/numpy.linalg.eigh.html) function.

In [ ]:
evals, evecs = np.linalg.eigh(covar)
# The eigenvalues are sorted in increasing order.
# Print the 10 most significant eigenvalues.
print(evals[-10:])
plt.close("eigvals")  # This is needed to rerun the code cell correctly
fig, ax = plt.subplots(num="eigvals")
ax.semilogy(evals)
ax.set_xlabel("Eigenvalue index")
ax.set_ylabel("Eigenvalue [nm$^2$]")

The first six eigenvalues are practically zero and correspond to the global rotation and translation, which were removed by the alignment.
The remaining 1740 modes represent internal motions of the protein, with increasing amplitude.
Many eigenvalues are very small, because they correspond to stiff internal coordinates:
bond lengths involving hydrogen atoms are even constrained in the simulation, and other bond lengths and valence angles can only vibrate with small amplitudes.
Only the modes with the largest eigenvalues describe collective motions of the protein with a large amplitude.
Because of the short simulation time, these modes are not yet sampled well, as shown in the next section.

## 4. Projection on a PCA mode

Once we have obtained the PCA modes (the eigenvectors), we can project the trajectory on one mode and follow the displacement along that mode as a function of time.
The following cell plots the projections on the three most significant modes, i.e. those with the largest eigenvalues.

In [ ]:
plt.close("projpca")  # This is needed to rerun the code cell correctly
fig, ax = plt.subplots(num="projpca")

# Compute the average geometry
av = xyz_eq.mean(axis=0)
# Plot projections on the three most significant principal modes.
# These are the last columns of evecs, because the eigenvalues are sorted in increasing order.
for i in range(3):
    proj = (xyz_eq - av) @ evecs[:, -i - 1]
    ax.plot(time_eq, proj, label=f"Mode {i + 1}")
ax.set_xlabel("Time [ps]")
ax.set_ylabel("Displacement [nm]")
ax.legend(loc=0)

The projections on the first two modes resemble half a period and a full period of a cosine function, respectively.
This so-called [cosine content](https://doi.org/10.1103/PhysRevE.65.031910) is typical for a random walk, showing that the MD simulation is far too short to completely explore the conformational space.
So far, the MD simulation essentially performed an undirected random walk in the conformational degrees of freedom of the protein.

## 5. Filtering the MD trajectory

One may visualize the motion along one or more modes by filtering the atomic positions, retaining only the displacements along the selected modes.
To do so, one projects the trajectory on the selected modes and reconstructs the atomic positions from these projections only.

In [ ]:
# The filtering algorithm.
# Motions along the three most significant modes are retained.
# Their eigenvectors are the columns of a (3N, 3) matrix.
modes = evecs[:, -3:]
# The projections on these modes form a (number of frames, 3) matrix.
projs = (xyz_eq - av) @ modes
# Reconstruct the atomic positions, using only these projections.
xyz_filter = av + projs @ modes.T

# Finally, we cast the result back into an MDTraj trajectory and visualize it.
traj_filter = mdtraj.Trajectory(xyz_filter.reshape(-1, traj.n_atoms, 3), traj.topology)
view_filter = nglview.show_mdtraj(traj_filter)
view_filter.add_representation("ball+stick", selection="protein")
view_filter

A careful inspection of the filtered trajectory should reveal the following:

- The average structure (to which the filtered modes are added) does not always make chemical sense.
  For example, methyl hydrogens overlap when a methyl group is freely rotating.
  More generally, PCA becomes meaningless for large conformational changes, because the average structure is then no longer representative.

- Most of the motion is close to the termini and some side chains, while the majority of the protein remains locked in place.

## 6. Extrapolation of a PCA mode

One may also visualize individual PCA modes by simply performing a linear extrapolation of the mode relative to the average structure.

In [ ]:
# The amplitude is the square root of the largest eigenvalue,
# i.e. the standard deviation of the projection on the first mode.
amplitude = np.sqrt(evals[-1])
# With evecs[:, -1], the last eigenvector,
# i.e. the first principal mode, is selected.
xyz_extra = np.outer(np.linspace(-amplitude, amplitude, 100), evecs[:, -1]) + av

# Again, we cast the result back into an MDTraj trajectory and visualize it.
traj_extra = mdtraj.Trajectory(xyz_extra.reshape(-1, traj.n_atoms, 3), traj.topology)
view_extra = nglview.show_mdtraj(traj_extra)
view_extra.add_representation("ball+stick", selection="protein")
view_extra

Also in this analysis, the individual frames are not necessarily realistic geometries.

## 7. Hinging mode in adenylate kinase

In the above examples, the principal modes were not very informative because the MD simulation was too short.
The villin headpiece is also not the best example to demonstrate collective modes.

The protein structures [2RGX](https://www.rcsb.org/structure/2rgx) and [2RH5](https://www.rcsb.org/structure/2rh5) represent two different conformations of one and the same protein: adenylate kinase from *Aquifex aeolicus*.
This is a phosphotransferase enzyme that catalyzes the interconversion of adenine nucleotides.
Basically, it is able to convert two molecules of adenosine diphosphate (ADP) to one adenosine monophosphate (AMP) and one adenosine triphosphate (ATP), and vice versa.
As such, it plays an important role in cellular energy homeostasis.
2RH5 describes the conformation of this enzyme in the absence of a substrate.
2RGX contains a substrate analogue, P1,P5-di(adenosine-5')pentaphosphate (Ap5A), and hence can be considered to describe the conformation of the enzyme in the presence of a substrate.
The main difference between both conformations resembles the action of Pac-Man: the enzyme opens up two lid regions in the absence of a ligand, and closes them once a ligand is present.

![pacman](pacman.png)

When using PDB files from the RCSB Protein Data Bank, it is common that some minor parts (atoms, residues, terminal groups, ...) are missing.
These issues can be fixed with a tool called PDBFixer.
General usage instructions can be found in the [PDBFixer manual](https://htmlpreview.github.io/?https://github.com/openmm/pdbfixer/blob/master/Manual.html).
It is also capable of a few other typical tasks, such as replacing non-standard residues or applying point mutations.

The following cell shows how to use PDBFixer in this case, with a for loop to fix both PDB files with the same code.
PDBFixer downloads the PDB files, applies the requested fixes and writes out the corrected PDB files.
All this will take a few seconds.
Missing residues at the termini of the chain are not added, because there is no experimental information about their conformation.
They would only add floppy tails to the protein, which are not relevant for the hinging motion.

The fixed PDB files will be suitable for superposition (next code cell) and as starting points for an MD simulation.

In [ ]:
for pdbid in "2rgx", "2rh5":
    print("Loading and fixing", pdbid)
    # Download the PDB file
    fixer = PDBFixer(pdbid=pdbid)
    # Remove all chains except the first
    nchain = len(list(fixer.topology.chains()))
    fixer.removeChains(range(1, nchain))
    # Remove all heterogens: ligands, ions and crystal water molecules
    fixer.removeHeterogens(keepWater=False)
    # Find missing residues, but discard those at the termini.
    # The keys of missingResidues are (chain index, residue index) tuples,
    # where the residue index is the position of the gap in the chain.
    fixer.findMissingResidues()
    nres = len(list(next(fixer.topology.chains()).residues()))
    fixer.missingResidues = {
        key: value for key, value in fixer.missingResidues.items() if 0 < key[1] < nres
    }
    # Add missing residues and atoms, including hydrogen atoms (at pH 7)
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.0)
    with open(f"{pdbid}_fixed.pdb", "w") as f:
        app.PDBFile.writeFile(fixer.topology, fixer.positions, f)

The following code shows how to load the two PDB files as single-frame trajectories in MDTraj.
They are aligned by minimizing the RMSD and then visualized with NGLView.
Because more residues were resolved at the C-terminus of 2RGX than of 2RH5, both structures do not have the same number of atoms.
The alignment is therefore based on the alpha carbons of the residues present in both structures.
The structure with the ligand analogue (2RGX) is shown in blue and the one without ligand (2RH5) in red.

In [ ]:
# Load trajectories
traj1 = mdtraj.load("2rgx_fixed.pdb")
traj2 = mdtraj.load("2rh5_fixed.pdb")
# Select the alpha carbons of residues 1 to 202, present in both structures.
ca1 = traj1.topology.select("name CA and resSeq 1 to 202")
ca2 = traj2.topology.select("name CA and resSeq 1 to 202")
# Superpose (in place)
traj2.superpose(traj1, atom_indices=ca2, ref_atom_indices=ca1)
# View
view = nglview.show_mdtraj(traj1)
view.add_component(traj2)
view[0].clear_representations()
view[1].clear_representations()
view[0].add_cartoon(color="blue")
view[1].add_cartoon(color="red")
view

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> 1. Identify the two lid regions in the overlay above, i.e. the parts of the enzyme that differ most between both structures.
>    Which residues do they contain?
>
> 2. Perform 2 nanoseconds of MD simulation using only one of the two states as initial structure.
>    Compute the RMSD as function of time, using the two crystal structures as references.
>    (As in the overlay above, a common selection of atoms is needed for this comparison.)
>    Also perform a PCA on this trajectory to identify and visualize the hinging mode.
>    Such a simulation is best submitted as a non-interactive job, as explained in [../03_noninteractive/01_noninteractive_notebook_on_hpc.ipynb](../03_noninteractive/01_noninteractive_notebook_on_hpc.ipynb).